## 📊 **Esplorazione Scalabile** - Definizione

L'**esplorazione scalabile** è l'insieme di tecniche e strategie per analizzare dataset **troppo grandi per essere ispezionati visivamente** (es. 100.000+ righe, 50+ colonne), dove non puoi semplicemente fare `df.head(20)` e "vedere" i pattern di sporcizia.

---

## 🔍 **Differenza chiave**

| Aspetto | Dataset piccolo (< 100 righe) | Dataset grande (> 10.000 righe) |
|---------|------------------------------|--------------------------------|
| **Esplorazione** | Guardi i dati direttamente | Usi statistiche e campionamento |
| **Pattern sporcizia** | Li vedi a occhio | Li inferisci da aggregazioni |
| **Outlier** | Li noti nello scrolling | Li trovi con IQR/Z-score |
| **NULL** | Conti a mano | Usi `isnull().sum()` |
| **Tempi** | Secondi | Minuti/ore (query ottimizzate) |

---

## 🛠️ **Tecniche di esplorazione scalabile**

### **1. Campionamento strategico**
```python
# Invece di guardare 1M righe, prendi un campione rappresentativo
df_sample = df.sample(1000, random_state=42)
df_sample['colonna'].value_counts()
```

### **2. Statistiche riassuntive invece di valori grezzi**
```python
# Non: df['colonna'].unique()  # potrebbe dare 10.000 valori
# Sì:
df['colonna'].value_counts(normalize=True).head(20)  # top 20 pattern
```

### **3. Profilazione automatica delle colonne**
```python
# Librerie dedicate
import pandas_profiling  # or ydata-profiling
profile = df.profile_report()
profile.to_file("report.html")
```

### **4. Pattern detection con regex su aggregati**
```sql
-- Invece di guardare ogni riga, conti i pattern
SELECT 
    CASE 
        WHEN colonna LIKE '%[a-z]%' THEN 'minuscole'
        WHEN colonna LIKE '%[A-Z]%' THEN 'MAIUSCOLE'
        ELSE 'numerico'
    END AS pattern,
    COUNT(*) AS n,
    COUNT(DISTINCT colonna) AS valori_distinti
FROM tabella
GROUP BY pattern;
```

### **5. Outlier detection su larga scala**
```python
# Invece di guardare valori "strani", usi metodi statistici
Q1 = df['valore'].quantile(0.25)
Q3 = df['valore'].quantile(0.75)
IQR = Q3 - Q1
outliers = df[(df['valore'] < Q1 - 1.5*IQR) | (df['valore'] > Q3 + 1.5*IQR)]
print(f"Outlier trovati: {len(outliers)} / {len(df)} ({len(outliers)/len(df)*100:.2f}%)")
```

### **6. Hash e fingerprinting per duplicati**
```python
# Trovare duplicati su dataset enorme senza confronto riga-riga
df['hash'] = df.apply(lambda x: hash(tuple(x)), axis=1)
duplicati = df[df.duplicated('hash', keep=False)]
```

---

## 📈 **Esempio pratico**

**Dataset piccolo (50 righe) - APPROCCIO DIRETTO:**
```python
df['Sito'].unique()
# Output: ['sud', 'Est', 'Nord', 'Centrale', 'est', 'Ovest', 'Sud', 'nord', ...]
# Vedi subito i pattern sporchi
```

**Dataset grande (500.000 righe) - APPROCCIO SCALABILE:**
```python
# Non puoi guardare 500.000 valori unici!
# Invece, fai:
sito_stats = df.groupby(df['Sito'].str.lower().str.strip()).agg({
    'Sito': ['count', 'nunique']
})
print(sito_stats)
# Output: vedi che 'nord' ha 3 varianti diverse (Nord, NORD, nord)
# Poi puoi decidere come normalizzare
```

---

## 🎯 **Perché è importante per te (geologo minerario)**

Nella realtà:
- Un progetto di esplorazione può avere **migliaia di sondaggi**
- Ogni sondaggio può avere **centinaia di campioni**
- I dati arrivano da **laboratori diversi** (formati inconsistenti)
- Non puoi aprire Excel e "guardare" i dati

Dovrai:
1. **Campionare** per capire i pattern
2. **Scrivere regole generali** di pulizia
3. **Validare** su campione prima di applicare a tutto
4. **Documentare** le scelte (per riproducibilità)

---

## 📝 **In sintesi**

| Concetto | Spiegazione semplice |
|----------|---------------------|
| **Esplorazione normale** | Guardi i dati con gli occhi |
| **Esplorazione scalabile** | Usi statistiche per capire i dati senza guardarli tutti |

---

Vuoi che ti prepari un **dataset di 5000 righe** per esercitarti con queste tecniche? Così provi concretamente la differenza! 🚀

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import random

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

np.random.seed(42)
random.seed(42)

n = 1000

# 1. ID Campione
id_campione = [f"CAMP-{i:04d}" for i in range(1, n+1)]
for i in random.sample(range(n), 50):
    id_campione[i] = id_campione[i].lower()

# 2. Sito
siti = ["Nord", "Sud", "Est", "Ovest"]
sito = [random.choice(siti) for _ in range(n)]
for i in random.sample(range(n), 80):
    sito[i] = sito[i].lower()

# 3. Profondità - uso una lista, non numpy array!
profondita = [round(random.uniform(10, 300), 1) for _ in range(n)]
# Aggiungi errori (ora funziona perché è una lista)
profondita[10] = "zero"
profondita[55] = "ND"
profondita[128] = -999
profondita[500] = 9999

# 4. Au_ppm - anche qui uso una lista
au_ppm = [round(np.random.lognormal(0, 1), 2) for _ in range(n)]
# Correggi valori >15
for i in range(n):
    if isinstance(au_ppm[i], (int, float)) and au_ppm[i] > 15:
        au_ppm[i] = 15
# Aggiungi errori
au_ppm[20] = "ND"
au_ppm[150] = "traccia"
au_ppm[300] = 99.9
au_ppm[600] = None

# 5. Data
data = [f"2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}" for _ in range(n)]
data[25] = "15/06/2024"
data[75] = "2024.09.20"
data[200] = "oggi"

# 6. Litologia
litologie = ["Granito", "Scisto", "Gneiss", "Marmo"]
litologia = [random.choice(litologie) for _ in range(n)]
for i in random.sample(range(n), 30):
    litologia[i] = litologia[i].lower()

# DataFrame
df_1000 = pd.DataFrame({
    'ID_Campione': id_campione,
    'Sito': sito,
    'Profondita_m': profondita,
    'Au_ppm': au_ppm,
    'Data': data,
    'Litologia': litologia
})

# Salva in SQLite
df_1000.to_sql('campioni_1000', conn, index=False, if_exists='replace')

print("✅ Dataset creato! 1000 righe con sporcizia realistica.")
print("\nPrime 10 righe:")
print(df_1000.head(10))
print(f"\nShape: {df_1000.shape}")

## 📋 Ricorda le tecniche che esploreremo:
## Tecnica	Quando la userai
* 1	df.sample() + value_counts()	Per vedere pattern senza guardare tutte le righe
* 2	df.describe() + df.info()	Panoramica rapida
* 3	df.isnull().sum()	Analisi NULL
* 4	IQR per outlier	Trovare valori anomali automaticamente
* 5	Regex su aggregati	Pattern detection su testo
* 6	df.duplicated()	Trovare duplicati
* 7	Correlazioni	Relazioni tra variabili

In [ ]:
# 1. Come SQLite vede la struttura
query = """
PRAGMA table_info(campioni_1000);
"""
print("📋 Struttura SQLite:")
df_tipi = pd.read_sql_query(query, conn)
print(df_tipi)
print("\n")

# 2. Verifica i tipi reali a livello di riga
print("Tipi effettivi (prime 5 righe):")
query_2 = """
    SELECT 
        ID_Campione AS ID_Campione, 
        typeof(Profondita_m) as tipo_prof, 
        typeof(Au_ppm) as tipo_au,
        Data AS Data
    FROM campioni_1000 LIMIT 10;
"""
df_tipi = pd.read_sql_query(query_2, conn)
print(df_tipi)
print("\n")

query_3 = """
SELECT 
  'Profondita_m' AS colonna,
  COUNT(*) AS totale,
  COUNT(CASE WHEN typeof(Profondita_m) = 'real' THEN 1 END) AS numeri_puri,
  COUNT(CASE WHEN typeof(Profondita_m) = 'text' AND Profondita_m IS NOT NULL THEN 1 END) AS testo,
  COUNT(CASE WHEN Profondita_m IS NULL THEN 1 END) AS null_sinceri
FROM campioni_1000
UNION ALL
SELECT 
  'Au_ppm',
  COUNT(*),
  COUNT(CASE WHEN typeof(Au_ppm) = 'real' THEN 1 END),
  COUNT(CASE WHEN typeof(Au_ppm) = 'text' AND Au_ppm IS NOT NULL THEN 1 END),
  COUNT(CASE WHEN Au_ppm IS NULL THEN 1 END)
FROM campioni_1000;
"""
df_tipi = pd.read_sql_query(query_3, conn)
print(df_tipi)
print("\n")

# 4. Crea una vista "pulita"
query_4 = """
CREATE VIEW campioni_validi AS
SELECT 
  ID_Campione,
  Sito,
  -- Converte Profondita_m solo se è un numero valido
  CASE 
    WHEN Profondita_m GLOB '[0-9]*' 
     AND Profondita_m NOT GLOB '*[a-zA-Z]*'
     AND CAST(Profondita_m AS REAL) BETWEEN 0 AND 500
    THEN CAST(Profondita_m AS REAL)
    ELSE NULL
  END AS Profondita_m_valida,
  
  -- Converte Au_ppm solo se è un numero valido
  CASE 
    WHEN Au_ppm GLOB '[0-9]*' 
     AND Au_ppm NOT GLOB '*[a-zA-Z]*'
     AND CAST(Au_ppm AS REAL) BETWEEN 0 AND 15
    THEN CAST(Au_ppm AS REAL)
    ELSE NULL
  END AS Au_ppm_valido,
  
  Data,
  Litologia
FROM campioni_1000;
"""
df_tipi = pd.read_sql_query(query_4, conn)
print(df_tipi)
print("\n")

# Verifica il risultato
query_5 = """
SELECT 
  COUNT(*) AS totale_righe,
  COUNT(Profondita_m_valida) AS profondita_valide,
  COUNT(Au_ppm_valido) AS au_validi,
  COUNT(*) - COUNT(Profondita_m_valida) AS profondita_scartate,
  COUNT(*) - COUNT(Au_ppm_valido) AS au_scartati
FROM campioni_validi;
"""
df_tipi = pd.read_sql_query(query_5, conn)
print(df_tipi)








